# Data Aggregation

The code trains N slightly different models, each of which is then tested on N different splits of the data. This generates N output files where the model has made predictions about a part of the dataset, so this code aggregates them into one dataframe and takes the mean of any duplicate entries (by checking whether `RID` (patient ID) and `delta_t` (timestep the prediction is made for) are both repeated in the same row). The end result is n = 1452 unique patient predictions.

In [11]:
BASE_DIR = Path('/home/staff7/fbsskr/Notebooks/Students/Oli/splits')

In [15]:
import pandas as pd
from pathlib import Path

# The common subdirectory structure leading to the CSV file
SUB_PATH = Path('eval_1/NMM/test_preds.csv')

# The range of the directories (0 to 19 inclusive, which means 20 directories)
NUM_REPETITIONS = 20

# List to store all individual dataframes
all_dfs = []

print(f"Starting data consolidation from {NUM_REPETITIONS} directories...")

# Loop through all 20 directories (rt0, rt1, ..., rt19)
for i in range(NUM_REPETITIONS):
    # Construct the directory name (e.g., 'rt0', 'rt1', etc.)
    dir_name = f'rt{i}'

    # Construct the full path to the CSV file
    full_path = BASE_DIR / dir_name / SUB_PATH

    # Check if the file exists before attempting to read it
    if full_path.is_file():
        try:
            # Read the CSV file into a pandas DataFrame
            df = pd.read_csv(full_path)

            # Add a column to identify the source of the data, which is useful
            # for ensemble analysis and tracking.
            df['source_run'] = dir_name

            # Append the loaded DataFrame to our list
            all_dfs.append(df)
            print(f"Successfully loaded data from: {full_path}")

        except Exception as e:
            print(f"Error reading file {full_path}: {e}")
    else:
        print(f"File not found: {full_path}. Skipping.")

# --- Consolidation ---
if all_dfs:
    # Concatenate all DataFrames in the list into a single DataFrame
    ensemble_df = pd.concat(all_dfs, ignore_index=True)
    
    # --- Data Transformation: Parsing 'Pred' Column ---
    print("\n--- Applying Prediction Column Transformation ---")
    
    # Check if 'Pred' column exists before attempting transformation
    if 'Pred' in ensemble_df.columns:
        # 1. Strip the outer brackets '[[', ']]' and any surrounding whitespace
        cleaned_pred_series = ensemble_df['Pred'].astype(str).str.strip('[] ')
        
        # 2. Split the cleaned string by ANY whitespace and expand into new columns (0, 1, 2)
        split_preds = cleaned_pred_series.str.split(expand=True).astype(float)
        
        # 3. Assign the split values to the requested new columns
        ensemble_df['CNPred'] = split_preds[0]
        ensemble_df['MCIPred'] = split_preds[1]
        ensemble_df['ADPred'] = split_preds[2]
        
        # 4. Drop the original 'Pred' column
        ensemble_df.drop(columns=['Pred'], inplace=True)
        
        print("Successfully created columns 'CNPred', 'MCIPred', 'ADPred' from 'Pred'.")
    else:
        print("Warning: 'Pred' column not found in consolidated data. Skipping transformation.")
    
    print("\n--- Consolidation Complete ---")
    
    # --- Ensemble Averaging & Mode Selection ---
    # We consolidate rows based on RID and delta_t.
    # Prediction columns (CNPred, MCIPred, ADPred) are averaged (mean).
    # Classification columns (FollowupDX, BaselineDX) use majority vote (mode).
    
    numerical_cols = ['CNPred', 'MCIPred', 'ADPred']
    classification_cols = ['BaselineDX', 'FollowupDX']
    grouping_keys = ['RID', 'delta_t']
    
    required_cols = grouping_keys + numerical_cols + classification_cols

    if all(col in ensemble_df.columns for col in required_cols):
        initial_rows = len(ensemble_df)
        
        # 1. Define the aggregation dictionary
        aggregation_dict = {}
        
        # Numerical columns: use mean for soft predictions
        for col in numerical_cols:
            aggregation_dict[col] = 'mean'
            
        # Classification columns: use mode (majority vote)
        # We use a lambda and .iloc[0] to ensure only the single, most frequent value 
        # is returned, even in the event of a tie (where pandas mode returns multiple values).
        for col in classification_cols:
            aggregation_dict[col] = lambda x: x.mode().iloc[0]

        # 2. Group by RID and delta_t, then apply the defined aggregation
        ensemble_df_aggregated = (
            ensemble_df.groupby(grouping_keys, as_index=False)
            .agg(aggregation_dict)
        )
        
        rows_aggregated = initial_rows - len(ensemble_df_aggregated)
        
        print("\n--- Ensemble Aggregation Complete ---")
        print(f"Initial total rows before aggregation: {initial_rows}")
        print(f"Number of rows consolidated (duplicates averaged/mode): {rows_aggregated}")
        
        ensemble_df = ensemble_df_aggregated
    else:
        missing_cols = [col for col in required_cols if col not in ensemble_df.columns]
        print(f"\nWarning: Skipping ensemble aggregation. Missing required columns: {missing_cols}")
    
    print(f"Total rows in the final ensemble DataFrame: {len(ensemble_df)}")
    
    # Print the count of unique RIDs in the final averaged DataFrame
    if 'RID' in ensemble_df.columns:
        print(f"Number of unique RIDs in the final ensemble: {ensemble_df['RID'].nunique()}")
        
    print(f"Columns: {ensemble_df.columns.tolist()}")

    # --- Output to New CSV ---
    # The filename reflects that the predictions have been averaged across runs.
    OUTPUT_FILENAME = 'combined_ensemble_predictions_averaged.csv'
    
    # Save the final DataFrame to a new CSV file
    ensemble_df.to_csv(OUTPUT_FILENAME, index=False)
    
    print(f"Final ensemble DataFrame successfully written to: {OUTPUT_FILENAME}")
else:
    print("\nNo dataframes were loaded. Please check the BASE_DIR and file paths.")

Starting data consolidation from 20 directories...
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt0/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt1/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt2/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt3/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt4/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt5/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt6/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/Students/Oli/splits/rt7/eval_1/NMM/test_preds.csv
Successfully loaded data from: /home/staff7/fbsskr/Notebooks/